# DATA Preparation


In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "30"  # To change accordingly tp yur threads

In [2]:
#imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [3]:
from utils.load_data import load_target_dataset
from utils.clinical_data_features import extract_cytogenetic_features, add_severity_features
from utils.molecular_data_features import process_molecular_data_effect, classify_impact

## Target Dataset

In [4]:
target_df = load_target_dataset("./target_train.csv")

## The molecular Dataset

In [5]:
# Clinical Data
df = pd.read_csv("X_train/clinical_train.csv")
df_eval = pd.read_csv("X_test/clinical_test.csv")

In [6]:
# Extract some features from chromosomes to both training and test sets
df = extract_cytogenetic_features(df)
df_eval = extract_cytogenetic_features(df_eval)

In [7]:
# Apply to both training and test sets
df = add_severity_features(df)
df_eval = add_severity_features(df_eval)

In [8]:
df.drop(columns=['CYTOGENETICS'], inplace=True)
df_eval.drop(columns=['CYTOGENETICS'], inplace=True)

## The gene dataset

In [9]:
df_mol = pd.read_csv("X_train/molecular_train.csv")
df_eval_mol = pd.read_csv("X_test/molecular_test.csv")

In [10]:
df_mol.head()

,ID,CHR,START,END,REF,ALT,GENE,PROTEIN_CHANGE,EFFECT,VAF,DEPTH
0,P100000,11,119149248.0,119149248.0,G,A,CBL,p.C419Y,non_synonymous_codon,0.0830,1308.0
1,P100000,5,131822301.0,131822301.0,G,T,IRF1,p.Y164*,stop_gained,0.0220,532.0
2,P100000,3,77694060.0,77694060.0,G,C,ROBO2,p.?,splice_site_variant,0.4100,876.0
3,P100000,4,106164917.0,106164917.0,G,T,TET2,p.R1262L,non_synonymous_codon,0.4300,826.0
4,P100000,2,25468147.0,25468163.0,ACGAAGAGGGGGTGTTC,A,DNMT3A,p.E505fs*141,frameshift_variant,0.0898,942.0


In [11]:
df_mol['EFFECT_LEVEL'] = df_mol['EFFECT'].apply(classify_impact)
df_eval_mol['EFFECT_LEVEL'] = df_eval_mol['EFFECT'].apply(classify_impact)

In [12]:
# Process the molecular data for both training and evaluation sets
df_mol_processed = process_molecular_data_effect(df_mol)
df_eval_mol_processed = process_molecular_data_effect(df_eval_mol)

## Dealing with missing values

To improve 

In [13]:
#We merge the processed molecular data with the clinical data
df = df.merge(df_mol_processed, on='ID', how='left')
#We fill NaN values with 0 for the new features
df.fillna(0, inplace=True)
df_eval = df_eval.merge(df_eval_mol_processed, on='ID', how='left')
#We fill NaN values with 0 for the new features
df_eval.fillna(0, inplace=True)

df.columns.to_series().to_csv("features.csv", index=False)

## Aligning and Splitting

In [14]:
#We allign the rows of the df with the target_df
df = df[df['ID'].isin(target_df['ID'])]

#We drop the ID column as it is not needed anymore
df.drop(columns=['ID', 'CENTER'], inplace=True)
df_eval.drop(columns=['ID', 'CENTER'], inplace=True)
target_df = target_df.drop(columns=['ID'])

In [15]:
# Now split
X_train, X_test, y_train, y_test = train_test_split(df, target_df, test_size=0.3, random_state=42)

In [16]:
from sksurv.util import Surv
y_train_struct = Surv.from_dataframe('OS_YEARS', 'OS_STATUS', y_train)
y_test_struct = Surv.from_dataframe('OS_YEARS', 'OS_STATUS', y_test)

# Machine Learning

In [17]:
#Importing the necessary libraries

import matplotlib.pyplot as plt
from sklearn.tree import plot_tree
from sklearn.model_selection import train_test_split
from sksurv.ensemble import RandomSurvivalForest
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis , IPCRidge
from sksurv.metrics import concordance_index_censored , concordance_index_ipcw
from sklearn.impute import SimpleImputer

### A simple model

In [39]:
#We use coxnet survival analysis
coxnet = CoxnetSurvivalAnalysis(
    n_alphas=200, 
    alphas=None, 
    alpha_min_ratio='auto',
    l1_ratio=0.3,
    penalty_factor=None,
    normalize=False,
    copy_X=True,
    tol=1e-09,
    max_iter=100000,
    verbose=True,
    fit_baseline_model=False,
      )
#We fit the model
coxnet.fit(X_train, y_train_struct)      
#We predict the survival function
y_pred = coxnet.predict(X_test)

#We calculate the concordance index
c_index = concordance_index_censored(y_test_struct['OS_YEARS'], y_test_struct['OS_STATUS'], y_pred)
print(f"Coxnet Concordance Index: {c_index[0]}")
#we also calculate the IPCW concordance index
ipcw_c_index = concordance_index_ipcw(survival_test=y_test_struct, survival_train=y_train_struct, estimate= y_pred )
print(f"Coxnet IPCW Concordance Index: {ipcw_c_index[0]}")

Coxnet Concordance Index: 0.7190871336903405
Coxnet IPCW Concordance Index: 0.6849356244839536


### More complex models

#### Component wise Gradient Boost

In [17]:
from sksurv.ensemble import ComponentwiseGradientBoostingSurvivalAnalysis
from sklearn.model_selection import KFold
from sksurv.metrics import concordance_index_ipcw
from scipy.stats import uniform
from joblib import Parallel, delayed
import numpy as np

n_iter = 25
n_splits = 10
n_jobs = n_splits  # nombre de folds parallèles, à ajuster selon CPU

best_params = None
best_score = float('-inf')

def eval_one_fold(train_index, val_index, params, X_train, y_train_struct):
    try:
        X_train_fold = X_train.iloc[train_index]
        X_val_fold = X_train.iloc[val_index]
        y_train_fold = y_train_struct[train_index]
        y_val_fold = y_train_struct[val_index]

        model = ComponentwiseGradientBoostingSurvivalAnalysis(**params, random_state=0)
        model.fit(X_train_fold, y_train_fold)
        y_pred = model.predict(X_val_fold)

        score = concordance_index_ipcw(
            survival_train=y_train_fold,
            survival_test=y_val_fold,
            estimate=y_pred
        )[0]
        return score
    except Exception as e:
        print(f"Fold failed with exception: {e}")
        return None

for i in range(n_iter):
    params = {
        "learning_rate": uniform.rvs(loc=0.001, scale=0.01),
        "n_estimators": int(uniform.rvs(loc=100, scale=1000)),
        "subsample": uniform.rvs(loc=0.3, scale=0.3)
    }

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=i)
    folds = list(kf.split(X_train, y_train_struct))

    fold_scores = Parallel(n_jobs=n_jobs)(
        delayed(eval_one_fold)(train_idx, val_idx, params, X_train, y_train_struct)
        for train_idx, val_idx in folds
    )
    
    fold_scores = [s for s in fold_scores if s is not None]

    if fold_scores:
        mean_score = np.mean(fold_scores)
        if mean_score > best_score:
            best_score = mean_score
            best_params = params

    print(f"[{i+1}/{n_iter}] Score moyen : {np.round(np.mean(fold_scores),4)}", flush=True)

print("\nMeilleurs paramètres :", best_params)
print("Score moyen sur validation croisée :", best_score)

best_model_CWGB = ComponentwiseGradientBoostingSurvivalAnalysis(**best_params, random_state=0)
best_model_CWGB.fit(X_train, y_train_struct)

y_pred_cwgboost = best_model_CWGB.predict(X_test)
ipcw_c_index_cwgboost = concordance_index_ipcw(
    survival_train=y_train_struct,
    survival_test=y_test_struct,
    estimate=y_pred_cwgboost
)

print(f"Componentwise CGBoost IPCW Concordance Index sur test : {ipcw_c_index_cwgboost[0]}")


[1/25] Score moyen : 0.5977
[2/25] Score moyen : 0.6735
[3/25] Score moyen : 0.662
[4/25] Score moyen : 0.6623
[5/25] Score moyen : 0.6618
[6/25] Score moyen : 0.6847
[7/25] Score moyen : 0.669
[8/25] Score moyen : 0.5549
[9/25] Score moyen : 0.6689
Fold failed with exception: censoring survival function is zero at one or more time points
[10/25] Score moyen : 0.6453
[11/25] Score moyen : 0.6622
[12/25] Score moyen : 0.6672
Fold failed with exception: censoring survival function is zero at one or more time points
[13/25] Score moyen : 0.6599
[14/25] Score moyen : 0.6508
[15/25] Score moyen : 0.6713
[16/25] Score moyen : 0.6107
[17/25] Score moyen : 0.634
Fold failed with exception: censoring survival function is zero at one or more time points
[18/25] Score moyen : 0.6322
[19/25] Score moyen : 0.6687
[20/25] Score moyen : 0.5613
Fold failed with exception: censoring survival function is zero at one or more time points
[21/25] Score moyen : 0.6758
[22/25] Score moyen : 0.6746
[23/25] Sc

In [ ]:
# Align the columns of df_eval with X_train
df_eval = df_eval.reindex(columns=X_train.columns)
df_eval.fillna(0, inplace= True)

#We predict the survival function for the test set
y_pred_test = best_model_CWGB.predict(df_eval)

In [ ]:
#We export the predictions to a csv file
df = pd.read_csv("X_test/clinical_test.csv")
predictions_df = pd.DataFrame({
    'ID': df['ID'],
    'risk_score': y_pred_test,
})
predictions_df.to_csv('submission_v8.csv', index=False)

#### Gradient Boost

In [19]:
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
from sksurv.ensemble import ComponentwiseGradientBoostingSurvivalAnalysis
from sklearn.model_selection import KFold
from sksurv.metrics import concordance_index_ipcw
from scipy.stats import uniform
from joblib import Parallel, delayed
import numpy as np
import os

# Nombre d'itérations de random search
n_iter = 25
n_splits = 10
n_jobs = n_splits # Nombre de folds traités en parallèle

best_score = float('-inf')
best_params = None

def eval_one_fold(train_index, val_index, params, X_train, y_train_struct):
    try:
        X_train_fold = X_train.iloc[train_index]
        X_val_fold = X_train.iloc[val_index]
        y_train_fold = y_train_struct[train_index]
        y_val_fold = y_train_struct[val_index]

        model = GradientBoostingSurvivalAnalysis(**params, random_state=0)
        model.fit(X_train_fold, y_train_fold)
        y_pred = model.predict(X_val_fold)

        score = concordance_index_ipcw(
            survival_train=y_train_fold,
            survival_test=y_val_fold,
            estimate=y_pred
        )[0]
        return score
    except Exception as e:
        print(f"Fold failed with exception: {e}")
        return None

for i in range(n_iter):
    # Hyperparamètres aléatoires
    params = {
        "learning_rate": uniform.rvs(loc=0.001, scale=0.1),
        "n_estimators": int(uniform.rvs(loc=400, scale=1200)),  
        "max_depth": int(uniform.rvs(loc=4, scale=14)),
        "min_samples_split": int(uniform.rvs(loc=10, scale=100)),
        "min_samples_leaf": int(uniform.rvs(loc=1, scale=10)),
        "subsample": uniform.rvs(loc=0.5, scale=0.45),
        "dropout_rate": uniform.rvs(loc=0.1, scale=0.5),
        "validation_fraction": 0.2
    }

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=i)
    folds = list(kf.split(X_train, y_train_struct))

    # Parallélisation ici
    fold_scores = Parallel(n_jobs=n_jobs)(
        delayed(eval_one_fold)(train_idx, val_idx, params, X_train, y_train_struct)
        for train_idx, val_idx in folds
    )

    # Supprime les folds échoués
    fold_scores = [s for s in fold_scores if s is not None]

    if fold_scores:
        mean_score = np.mean(fold_scores)
        if mean_score > best_score:
            best_score = mean_score
            best_params = params

    print(f"[{i+1}/{n_iter}] Score moyen : {np.round(np.mean(fold_scores), 4)}", flush=True)

# Affichage des meilleurs paramètres
print("\nMeilleurs paramètres :", best_params)
print("Score moyen sur validation croisée :", best_score)

# Ré-entraîner avec les meilleurs paramètres
best_model_CGB = GradientBoostingSurvivalAnalysis(**best_params, random_state=0)
best_model_CGB.fit(X_train, y_train_struct)

# Évaluation finale
y_pred_cgboost = best_model_CGB.predict(X_test)
ipcw_c_index_cgboost = concordance_index_ipcw(
    survival_train=y_train_struct,
    survival_test=y_test_struct,
    estimate=y_pred_cgboost
)

print(f"CGBoost IPCW Concordance Index sur test : {ipcw_c_index_cgboost[0]}")


[1/25] Score moyen : 0.6795
[2/25] Score moyen : 0.6875
[3/25] Score moyen : 0.685
[4/25] Score moyen : 0.6781
[5/25] Score moyen : 0.6893
[6/25] Score moyen : 0.6899
[7/25] Score moyen : 0.6869
[8/25] Score moyen : 0.6929
[9/25] Score moyen : 0.6918
Fold failed with exception: censoring survival function is zero at one or more time points
[10/25] Score moyen : 0.6958
[11/25] Score moyen : 0.6796
[12/25] Score moyen : 0.6736
Fold failed with exception: censoring survival function is zero at one or more time points
[13/25] Score moyen : 0.679
[14/25] Score moyen : 0.6841
[15/25] Score moyen : 0.6902
[16/25] Score moyen : 0.6916
[17/25] Score moyen : 0.6929
Fold failed with exception: censoring survival function is zero at one or more time points
[18/25] Score moyen : 0.6833
[19/25] Score moyen : 0.681
[20/25] Score moyen : 0.6991
Fold failed with exception: censoring survival function is zero at one or more time points
[21/25] Score moyen : 0.6654
[22/25] Score moyen : 0.6901
[23/25] Sc

The best model is : 




In [20]:
# Align the columns of df_eval with X_train
df_eval = df_eval.reindex(columns=X_train.columns)
df_eval.fillna(0, inplace= True)

#We predict the survival function for the test set
y_pred_test = best_model_CGB.predict(df_eval)

In [21]:
#We export the predictions to a csv file
df = pd.read_csv("X_test/clinical_test.csv")

predictions_df = pd.DataFrame({
    'ID': df['ID'],
    'risk_score': y_pred_test,
})
predictions_df.to_csv('submission_v11.csv', index=False)

#### Survival Forest

In [21]:
from sksurv.ensemble import RandomSurvivalForest
from sklearn.model_selection import KFold
from sksurv.metrics import concordance_index_ipcw
from scipy.stats import uniform, randint
from joblib import Parallel, delayed
import numpy as np

n_iter = 1
n_splits = 2
n_jobs = n_splits   # ajuster selon CPU

best_params = None
best_score = float('-inf')

def eval_one_fold(train_idx, val_idx, params, X_train, y_train_struct):
    try:
        X_train_fold = X_train.iloc[train_idx]
        X_val_fold = X_train.iloc[val_idx]
        y_train_fold = y_train_struct[train_idx]
        y_val_fold = y_train_struct[val_idx]

        model = RandomSurvivalForest(**params, random_state=0)
        model.fit(X_train_fold, y_train_fold)
        y_pred = model.predict(X_val_fold)

        score = concordance_index_ipcw(
            survival_train=y_train_fold,
            survival_test=y_val_fold,
            estimate=y_pred
        )[0]
        return score
    except Exception as e:
        print(f"Fold failed with exception: {e}")
        return None

for i in range(n_iter):
    params = {
        "n_estimators": 10000,
        "max_depth": 10,
        "min_samples_split": 8,
        "min_samples_leaf": 5,
        "max_features": ['sqrt', 'log2', None][np.random.choice(3)],
        "min_weight_fraction_leaf": 0,
        "min_samples_leaf": 5,
        "n_jobs": -1
    }

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=i)
    folds = list(kf.split(X_train, y_train_struct))

    fold_scores = Parallel(n_jobs=n_jobs)(
        delayed(eval_one_fold)(train_idx, val_idx, params, X_train, y_train_struct)
        for train_idx, val_idx in folds
    )
    fold_scores = [s for s in fold_scores if s is not None]

    if fold_scores:
        mean_score = np.mean(fold_scores)
        if mean_score > best_score:
            best_score = mean_score
            best_params = params

    print(f"[{i+1}/{n_iter}] Score moyen : {np.round(np.mean(fold_scores), 4)}", flush=True)

print("\nMeilleurs paramètres :", best_params)
print("Score moyen sur validation croisée :", best_score)

best_model_rf = RandomSurvivalForest(**best_params, random_state=0)
best_model_rf.fit(X_train, y_train_struct)

y_pred_rf = best_model_rf.predict(X_test)
ipcw_c_index_rf = concordance_index_ipcw(
    survival_train=y_train_struct,
    survival_test=y_test_struct,
    estimate=y_pred_rf
)

print(f"Random Forest IPCW Concordance Index sur test : {ipcw_c_index_rf[0]}")

[1/1] Score moyen : 0.6905

Meilleurs paramètres : {'n_estimators': 10000, 'max_depth': 10, 'min_samples_split': 8, 'min_samples_leaf': 5, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0, 'n_jobs': -1}
Score moyen sur validation croisée : 0.6904879971409471
Random Forest IPCW Concordance Index sur test : 0.6890651398786258


In [22]:
# Align the columns of df_eval with X_train
df_eval = df_eval.reindex(columns=X_train.columns)

#We predict the survival function for the test set
y_pred_test = best_model_rf.predict(df_eval)

In [23]:
#We export the predictions to a csv file

df = pd.read_csv("X_test\clinical_test.csv")

predictions_df = pd.DataFrame({
    'ID': df['ID'],
    'risk_score': y_pred_test,
})
predictions_df.to_csv('submission_v_2_2.csv', index=False)

<>:3: SyntaxWarning: invalid escape sequence '\c'
<>:3: SyntaxWarning: invalid escape sequence '\c'
C:\Users\marec\AppData\Local\Temp\ipykernel_30616\4220546691.py:3: SyntaxWarning: invalid escape sequence '\c'
  df = pd.read_csv("X_test\clinical_test.csv")


## Pytorch deep learning